# Text Preprocessing

Before we can train a language model, we need to convert raw text into a format the model can understand. This notebook covers the first step: loading text data and splitting it into tokens.

**What we'll do:**
- Download a sample text file
- Split text into individual tokens using regex
- Handle punctuation and special characters

## Loading the Data

We'll use a short story called "The Verdict" as our training text. It's small enough to work with quickly but has enough variety to be interesting.

In [1]:
import os
import requests

# Download the text file if we don't have it
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)
    print("Downloaded the-verdict.txt")
else:
    print("File already exists")

File already exists


In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text)}")
print(f"\nFirst 100 characters:\n{raw_text[:100]}")

Total characters: 20479

First 100 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## Splitting Text into Tokens

Tokenization means breaking text into smaller pieces. The simplest approach is splitting on whitespace, but that doesn't handle punctuation well.

For example, "Hello, world." split on whitespace gives us `["Hello,", "world."]` - the punctuation is stuck to the words.

We want: `["Hello", ",", "world", "."]`

In [3]:
import re

# Start simple: split on whitespace
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print("Split on whitespace:")
print(result)

Split on whitespace:
['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
# Better: also split on commas and periods
result = re.split(r'([,.]|\s)', text)
print("Split on whitespace, commas, periods:")
print(result)

Split on whitespace, commas, periods:
['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [5]:
# Clean up empty strings and whitespace
result = [item.strip() for item in result if item.strip()]
print("After cleanup:")
print(result)

After cleanup:
['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


## Complete Tokenization Pattern

Now let's handle all common punctuation marks. The regex pattern below splits on:
- Basic punctuation: `, . : ; ? ! " ( ) '`
- Double dashes: `--`
- Whitespace

In [6]:
text = "Hello, world. Is this-- a test?"

# The full pattern
pattern = r'([,.:;?_!"()\']|--|\s)'

result = re.split(pattern, text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


## Tokenizing the Full Text

Let's apply this to our entire training text.

In [7]:
pattern = r'([,.:;?_!"()\']|--|\s)'

preprocessed = re.split(pattern, raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(f"Total tokens: {len(preprocessed)}")
print(f"\nFirst 30 tokens:\n{preprocessed[:30]}")

Total tokens: 4690

First 30 tokens:
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Building the Vocabulary

A vocabulary maps each unique token to an integer ID. Models work with numbers, not strings.

We sort the tokens alphabetically so the mapping is consistent across runs.

In [8]:
# Get unique tokens and sort them
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1130


In [9]:
# Create the token-to-ID mapping
vocab = {token: idx for idx, token in enumerate(all_words)}

# Look at the first few entries
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


## Summary

We now have:
1. A way to split text into tokens using regex
2. A vocabulary that maps each unique token to an integer

Next step: Build a tokenizer class that can encode text to IDs and decode IDs back to text.